# 3.1 · Test Driven Development

In this recipe, you will build a test suite for an IBM® [Granite](https://www.ibm.com/granite) function-calling agent, in the spirit of Test-Driven Development (TDD): you write down what the agent *should* do for a set of inputs, then check every run against it.

You will first build a small agent that answers weather and stock price questions with two tools. Then you will write checks for its **trajectory** (did it call the right tools with the right arguments?) and its **response** (does the final answer contain the expected facts?), run single-turn and multi-turn conversations through it, and reduce the results to a pass rate, a latency and a token count you can track over time.

# Prerequisites

Before testing your agent, you'll need to set up the environment and create a basic function-calling agent. The following prerequisites walk you through the necessary setup.

## Set up your environment

While you can choose from several tools, this recipe is best suited for a Jupyter Notebook. Jupyter Notebooks are widely used within data science to combine code with various data sources such as text, images and data visualizations. 

You can run this notebook in [Colab](https://colab.research.google.com/drive/1kZl5o2oDJEQ72kLedaetQ7UXqFU9BEW2?usp=sharing), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md). 

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

Note, this notebook is compatible with Python 3.12 and well as Python 3.11, the default in Colab at the time of publishing this recipe. To check your python version, you can run the `!python --version` command in a code cell.

## Set up a Granite AI model instance

This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. It works with either:

- [Ollama](https://ollama.com), running locally, with the `granite4.2:3b` model pulled, or
- the [Replicate](https://replicate.com) cloud service. See [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb) for information on getting ready to use Replicate, and obtain an [API token](https://replicate.com/account/api-tokens).

If `REPLICATE_API_TOKEN` is set, the notebook uses Replicate; otherwise it uses the local Ollama model.

## Install relevant libraries and set up credentials and the Granite model

We'll need a few libraries for this recipe. We will be using LangGraph and LangChain libraries to use Granite.

In [ ]:
# Install / upgrade dependencies used in this notebook.
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langgraph-prebuilt \
    langchain \
    langchain_ollama \
    langfuse \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

Now we will create the LangChain object to use the Granite model.

In [ ]:
import os

import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

In [ ]:
llm_granite = get_llm()
print(f"Connected via {type(llm_granite).__name__}, model={llm_granite.model!r}")

## Define the tools

We define two functions to be used as tools by our agent. These functions can use real web APIs if you obtain the necessary API keys. If you are unable to get the API keys, the tools below will respond with a fixed, predetermined value for demonstration purposes.

- **`get_stock_price`**: Uses an `AV_STOCK_API_KEY` from [Alpha Vantage](https://www.alphavantage.co/support/#api-key)
- **`get_current_weather`**: Uses a `WEATHER_API_KEY` from [OpenWeather](https://home.openweathermap.org/users/sign_up)

**Store these private keys in a separate `.env` file in the same level of your directory as this notebook.**

In [ ]:
AV_STOCK_API_KEY = convert_to_secret_str(get_env_var("AV_STOCK_API_KEY", "unset"))

WEATHER_API_KEY = convert_to_secret_str(get_env_var("WEATHER_API_KEY", "unset"))

The function's docstring and type information are important for generating proper tool information, as this will be the basis of the tool description provided to the model.

In [ ]:
def get_stock_price(ticker: str, date: str) -> dict:
    """
    Retrieves the lowest and highest stock prices for a given ticker and date.

    Args:
        ticker: The stock ticker symbol, for example, "IBM".
        date: The date in "YYYY-MM-DD" format for which you want to get stock prices.

    Returns:
        A dictionary containing the low and high stock prices on the given date.
    """
    print(f"Getting stock price for {ticker} on {date}")

    apikey = AV_STOCK_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return {
            "low": "245.4500",
            "high": "249.0300"
        }

    try:
        stock_url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={ticker}&apikey={apikey}"
        stock_data = requests.get(stock_url)
        data = stock_data.json()
        stock_low = data["Time Series (Daily)"][date]["3. low"]
        stock_high = data["Time Series (Daily)"][date]["2. high"]
        return {
            "low": stock_low,
            "high": stock_high
        }
    except Exception as e:
        print(f"Error fetching stock data: {e}")
        return {
            "low": "none",
            "high": "none"
        }

The `get_current_weather` function retrieves the real-time weather in a given location using the Current Weather Data API via [OpenWeather](https://openweathermap.org/api). 

In [ ]:
def get_current_weather(location: str) -> dict:
    """
    Fetches the current weather for a given location.

    Args:
        location: The name of the city for which to retrieve the weather information.

    Returns:
        A dictionary containing weather information such as temperature in celsius, weather description, and humidity.
    """
    print(f"Getting current weather for {location}")
    apikey=WEATHER_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return {
            "description": "thunderstorms",
            "temperature": 25.3,
            "humidity": 94
        }

    try:
        # API request to fetch weather data
        weather_url = f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid={apikey}&units=metric"
        weather_data = requests.get(weather_url)
        data = weather_data.json()
        # Extracting relevant weather details
        weather_description = data["weather"][0]["description"]
        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]

        # Returning weather details
        return {
            "description": weather_description,
            "temperature": temperature,
            "humidity": humidity
        }
    except Exception as e:
        print(f"Error fetching weather data: {e}")
        return {
            "description": "none",
            "temperature": "none",
            "humidity": "none"
        }

## Set up Langfuse

Every test run in this notebook is traced in [Langfuse](https://langfuse.com), so when a test fails you can open its trace and see which model generations and tool calls led there. At the end, the whole test set also runs as a Langfuse **experiment**, with the test checks recorded as scores.

Use the same Langfuse instance and `.env` keys as in [`02_Agent_Tracing_Observability.ipynb`](02_Agent_Tracing_Observability.ipynb):

```dotenv
LANGFUSE_SECRET_KEY=sk-lf-xxxxx
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxx
LANGFUSE_HOST=http://localhost:3000
```

In [ ]:
from IPython.display import HTML, display
from langfuse import Langfuse

langfuse_client = Langfuse(
    public_key=get_env_var("LANGFUSE_PUBLIC_KEY", "unset"),
    secret_key=get_env_var("LANGFUSE_SECRET_KEY", "unset"),
    host=get_env_var("LANGFUSE_HOST", "unset")
)

# Without a reachable Langfuse the tests still run; only the Langfuse datasets, experiments and links are skipped.
try:
    LANGFUSE_AVAILABLE = langfuse_client.auth_check()
except Exception as e:
    print(f"Langfuse unavailable ({e}); tests will run without Langfuse datasets, experiments and links")
    LANGFUSE_AVAILABLE = False
print(f"Langfuse available: {LANGFUSE_AVAILABLE}")


def show_link(label: str, url: str | None) -> None:
    """Display a clickable link, or say why there isn't one."""
    if url:
        display(HTML(f'<p>{label}: <a href="{url}" target="_blank">{url}</a></p>'))
    else:
        print(f"{label}: unavailable")


def show_trace_link(trace_id: str, label: str = "View this trace in Langfuse") -> None:
    """Display a link to the trace in the Langfuse UI."""
    if not LANGFUSE_AVAILABLE:
        return
    show_link(label, langfuse_client.get_trace_url(trace_id=trace_id))

## Create the agent

LangChain provides a convenient method to create a function-calling agent. You just need to provide the model and the list of tools. For a detailed walkthrough of building agents from scratch with LangGraph, see the [Function Calling Agent recipe](01_Function_Calling_Agent.ipynb).

We use `create_agent` from LangChain to quickly build our function-calling agent. Not specifying the `prompt` argument means the agent will use the default system prompt for tool calling.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.graph.state import CompiledStateGraph

tools = [get_stock_price, get_current_weather]

agent: CompiledStateGraph = create_agent(
    model=llm_granite,
    tools=tools,
)

Let's verify the agent works with a simple query:

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class State(TypedDict, total=False):
    """Agent state that holds the conversation messages."""
    messages: Annotated[list[AnyMessage], add_messages]

def run_agent(graph: CompiledStateGraph, user_input: str):
    """Helper function to run the agent and display the conversation."""
    user_message = HumanMessage(user_input)
    print(user_message.pretty_repr())
    input_state = State(messages=[user_message])
    for event in graph.stream(input_state):
        for value in event.values():
            print(value["messages"][-1].pretty_repr())

# Test the agent with a simple query
run_agent(agent, "What is the weather in Miami?")

# Testing Your Agent

Now that we have a working function-calling agent, let's create a structured test framework to evaluate its behavior. 

Testing AI agents is analogous to Test-Driven Development (TDD) in traditional software engineering. Just as TDD provides confidence that your code works as expected, agent testing ensures your AI behaves reliably and consistently. This is a **necessity for productionizing your agent**—without proper testing, you cannot confidently deploy updates or compare model alternatives.

**Why test your agent?**

- **Catch regressions early** when you update prompts, tool schemas, or models
- **Compare alternatives objectively** (e.g., model A vs. model B)
- **Ship with confidence** because you know core use cases still pass
- **Faster debugging** by reproducing issues with specific test cases
- **Documentation** as your test cases become living examples of expected behavior

In this notebook, we'll focus on testing **tool-calling agents** and implement:
1. Evaluation helpers for tool-call trajectories and responses
2. Structured test cases for single-turn and multi-turn interactions, stored as Langfuse datasets
3. Summary metrics tracking
4. Langfuse experiments that run the datasets and record the checks as scores

## Step 1: Define data structures and evaluation helpers

Before running tests, we define how to evaluate agent outputs. This follows TDD principles: **write your assertions first**.

For tool-calling agents, we care about two types of evaluation:

- **Trajectory evaluation**: Did the agent call the right tools with the right parameters? Use exact matching when mistakes are risky (e.g., tools that write or delete data).
- **Response evaluation**: Did the agent's final response contain the expected content? For deterministic tests, substring matching is fast and easy to debug. For flexible responses, consider LLM-as-a-judge or semantic similarity.

In [ ]:
import time
from dataclasses import dataclass, field
from typing import Any


@dataclass
class ToolCall:
    """Represents a single tool call made by the agent."""
    tool_name: str
    tool_parameters: dict[str, Any]


@dataclass
class AgentTestResult:
    """The output of an agent test run, including tool calls and metrics."""
    tool_calls: list[ToolCall] = field(default_factory=list)
    final_response: str = ""
    latency_ms: float = 0.0
    prompt_tokens: int = 0
    response_tokens: int = 0
    total_tokens: int = 0
    trace_id: str = ""


def trajectory_match(actual: list[ToolCall], expected: list[dict[str, Any]]) -> bool:
    """Check if actual tool calls exactly match expected.

    Use exact match when mistakes are risky (e.g., tools that write or delete data).
    """
    actual_norm = [{"tool_name": c.tool_name, "tool_parameters": c.tool_parameters} for c in actual]
    return actual_norm == expected


def response_match(actual: str, expected_contains: str | list[str]) -> bool:
    """Check if actual response contains every expected substring (case-insensitive).

    For deterministic tests, a simple substring check is fast and easy to debug.
    For flexible responses, consider LLM-as-a-judge or semantic similarity.
    """
    expected = [expected_contains] if isinstance(expected_contains, str) else expected_contains
    return all(e.lower() in actual.lower() for e in expected)

## Step 2: Create a test runner for the agent

We wrap our LangGraph agent in a test runner that captures tool calls, responses, and metrics for evaluation. The runner streams through the agent execution, capturing:
- Tool calls from AI messages (tool name and arguments)
- Final text response (when no tool calls are requested)
- Performance metrics (latency, and the token usage the model reports for every call it makes)

The optional `thread_id` lets a test continue an earlier conversation; we use it for the multi-turn tests in Step 5.

Each run is also traced in Langfuse with the run name `TDD` and the tag `tdd`, and the result keeps its `trace_id`, so you can open the trace of any failing test. The turns of a multi-turn test share a Langfuse session (their `thread_id`), so the whole conversation shows up together.

In [ ]:
from langchain_core.messages import AIMessage
from langfuse.langchain import CallbackHandler


def run_agent_for_test(
    graph: CompiledStateGraph,
    user_input: str,
    thread_id: str | None = None,
    *,
    new_trace: bool = True,
) -> AgentTestResult:
    """Run the agent and collect results for testing purposes.

    Every run is traced in Langfuse under the run name "TDD".
    Pass a `thread_id` to continue a conversation on a graph that has a checkpointer;
    its turns are grouped in one Langfuse session.
    Set `new_trace=False` to nest the run in the current Langfuse trace (as in an experiment).
    """
    start = time.time()
    tool_calls_made: list[ToolCall] = []
    final_response = ""
    prompt_tokens = 0
    response_tokens = 0

    user_message = HumanMessage(user_input)
    input_state = State(messages=[user_message])

    if new_trace:
        trace_id = langfuse_client.create_trace_id()
        langfuse_handler = CallbackHandler(trace_context={"trace_id": trace_id})
    else:
        trace_id = langfuse_client.get_current_trace_id() or ""
        langfuse_handler = CallbackHandler()

    config = {
        "callbacks": [langfuse_handler],
        "run_name": "TDD",
        "metadata": {"langfuse_tags": ["tdd"]},
    }
    if thread_id:
        config["configurable"] = {"thread_id": thread_id}
        config["metadata"]["langfuse_session_id"] = thread_id

    # Stream through the agent execution; each event holds the messages a node just added
    for event in graph.stream(input_state, config=config):
        for value in event.values():
            for message in (value or {}).get("messages", []):
                if not isinstance(message, AIMessage):
                    continue

                # Capture tool calls from AI messages
                for tc in message.tool_calls:
                    tool_calls_made.append(ToolCall(
                        tool_name=tc['name'],
                        tool_parameters=tc['args']
                    ))
                # Capture final text response (when no tool calls)
                if message.content and not message.tool_calls:
                    final_response = message.content
                # Add up the tokens the model reports for each call (system prompt and tool schemas included)
                if message.usage_metadata:
                    prompt_tokens += message.usage_metadata["input_tokens"]
                    response_tokens += message.usage_metadata["output_tokens"]

    latency_ms = (time.time() - start) * 1000

    return AgentTestResult(
        tool_calls=tool_calls_made,
        final_response=final_response,
        latency_ms=latency_ms,
        prompt_tokens=prompt_tokens,
        response_tokens=response_tokens,
        total_tokens=prompt_tokens + response_tokens,
        trace_id=trace_id,
    )

## Step 3: Define the evaluation datasets

Our test cases are an **evaluation dataset**: a fixed set of inputs, each with its expected outcome. We define two, one per kind of test.

Each single-turn test case includes:
- **Input**: The user query to be processed
- **Expected tool calls**: The tools and parameters the agent should use
- **Expected response**: One or more substrings the final response should contain

We cover key scenarios for our weather and stock price tools. Note how we test both basic functionality and parameter variations to ensure the agent properly extracts and normalizes information from queries.

To keep the tests deterministic, we run them against the tools' fixed demonstration values instead of the live APIs: live weather changes every hour, and Alpha Vantage's daily series only covers recent trading days. With fixed tool results, the response checks can assert on the values the tools return (for example the IBM `low` of `245.45`), not only on words the question already contains.

In [ ]:
# The tools read these keys each time they're called, so from here on they return their fixed values.
AV_STOCK_API_KEY = convert_to_secret_str("unset")
WEATHER_API_KEY = convert_to_secret_str("unset")

In [ ]:
AGENT_TESTS = [
    {
        "name": "weather_query",
        "input": "What is the weather in Miami?",
        "expected_tool_calls": [
            {"tool_name": "get_current_weather", "tool_parameters": {"location": "Miami"}}
        ],
        "expected_response_contains": ["Miami", "25.3"]
    },
    {
        "name": "stock_price_query",
        "input": "What were the IBM stock prices on September 10, 2026?",
        "expected_tool_calls": [
            {"tool_name": "get_stock_price", "tool_parameters": {"ticker": "IBM", "date": "2026-09-10"}}
        ],
        "expected_response_contains": ["IBM", "245.45"]
    },
    {
        "name": "weather_different_city",
        "input": "Tell me the current weather in New York",
        "expected_tool_calls": [
            {"tool_name": "get_current_weather", "tool_parameters": {"location": "New York"}}
        ],
        "expected_response_contains": ["New York", "25.3"]
    },
    {
        "name": "stock_different_ticker",
        "input": "Get me the stock price for AAPL on September 15, 2026",
        "expected_tool_calls": [
            {"tool_name": "get_stock_price", "tool_parameters": {"ticker": "AAPL", "date": "2026-09-15"}}
        ],
        "expected_response_contains": ["AAPL", "245.45"]
    },
]

### Multi-turn test cases

Real conversations don't happen in isolation. Users ask follow-up questions, reference previous context, and switch topics mid-conversation. **Multi-turn tests** verify that the agent can handle sequential queries in a conversation context: each test case is a conversation, with the same expectations for every turn.

These tests check:
- **Tool type switching**: Can the agent transition between different tool types within a conversation?
- **Contextual reference handling**: Does the agent understand follow-up questions like "How about in Tokyo?" without explicit mention of "weather", or "What about AAPL on the same date?", which can't be answered without the previous turn?

In [ ]:
MULTI_TURN_TESTS = [
    {
        "name": "weather_then_stock",
        "turns": [
            {
                "input": "What is the weather in Boston?",
                "expected_tool_calls": [
                    {"tool_name": "get_current_weather", "tool_parameters": {"location": "Boston"}}
                ],
                "expected_response_contains": ["Boston", "25.3"]
            },
            {
                "input": "Now tell me the IBM stock price on September 10, 2026",
                "expected_tool_calls": [
                    {"tool_name": "get_stock_price", "tool_parameters": {"ticker": "IBM", "date": "2026-09-10"}}
                ],
                "expected_response_contains": ["IBM", "245.45"]
            },
            {
                # Only answerable from the previous turn: the date isn't repeated
                "input": "What about AAPL on the same date?",
                "expected_tool_calls": [
                    {"tool_name": "get_stock_price", "tool_parameters": {"ticker": "AAPL", "date": "2026-09-10"}}
                ],
                "expected_response_contains": ["AAPL", "245.45"]
            }
        ]
    },
    {
        "name": "multiple_weather_queries",
        "turns": [
            {
                "input": "What's the weather like in London?",
                "expected_tool_calls": [
                    {"tool_name": "get_current_weather", "tool_parameters": {"location": "London"}}
                ],
                "expected_response_contains": ["London", "25.3"]
            },
            {
                "input": "How about in Tokyo?",
                "expected_tool_calls": [
                    {"tool_name": "get_current_weather", "tool_parameters": {"location": "Tokyo"}}
                ],
                "expected_response_contains": ["Tokyo", "25.3"]
            },
            {
                "input": "And what about Paris?",
                "expected_tool_calls": [
                    {"tool_name": "get_current_weather", "tool_parameters": {"location": "Paris"}}
                ],
                "expected_response_contains": ["Paris", "25.3"]
            },
        ]
    }
]

### Store the datasets in Langfuse

The Python lists above are the source of truth: they live with the notebook, under version control. We also store them as **Langfuse datasets**, so Step 7 can run them as experiments, and every experiment run is linked to the dataset in the Langfuse UI for side-by-side comparison.

| Langfuse dataset | Item input | Item expected output |
| :--- | :--- | :--- |
| `tdd-single-turn` | The question | The expected tool calls and response substrings |
| `tdd-multi-turn` | The list of turns | The expected tool calls and response substrings, per turn |

Each item gets a fixed id built from the test name, so re-running this cell updates the items instead of duplicating them.

In [ ]:
SINGLE_TURN_DATASET = "tdd-single-turn"
MULTI_TURN_DATASET = "tdd-multi-turn"


def expectations(case: dict) -> dict:
    """The expected outcome of one test or turn."""
    return {
        "tool_calls": case["expected_tool_calls"],
        "response_contains": case["expected_response_contains"],
    }


def create_langfuse_dataset(name: str, description: str, items: list[dict]) -> None:
    """Create the dataset if it doesn't exist yet, then upsert its items."""
    try:
        langfuse_client.get_dataset(name)
    except Exception:
        langfuse_client.create_dataset(name=name, description=description)

    for item in items:
        # A fixed id makes this an upsert: re-running the cell doesn't duplicate items
        langfuse_client.create_dataset_item(dataset_name=name, id=f"{name}-{item['metadata']['name']}", **item)
    print(f"Dataset {name!r}: {len(items)} items")


if LANGFUSE_AVAILABLE:
    create_langfuse_dataset(
        SINGLE_TURN_DATASET,
        "Single-turn TDD tests for the weather and stock price agent",
        [
            {"input": test["input"], "expected_output": expectations(test), "metadata": {"name": test["name"]}}
            for test in AGENT_TESTS
        ],
    )
    create_langfuse_dataset(
        MULTI_TURN_DATASET,
        "Multi-turn TDD tests for the weather and stock price agent",
        [
            {
                "input": {"turns": [turn["input"] for turn in test["turns"]]},
                "expected_output": [expectations(turn) for turn in test["turns"]],
                "metadata": {"name": test["name"]},
            }
            for test in MULTI_TURN_TESTS
        ],
    )

## Step 4: Run single-turn tests

Single-turn tests verify basic functionality for each tool. We run each test case through the agent and evaluate both:
- **Trajectory**: Did the agent call the correct tool(s)?
- **Response**: Did the final answer contain the expected information?

In [ ]:
def run_single_turn_tests(test_graph: CompiledStateGraph, tests: list[dict]) -> list[dict]:
    """Run all single-turn tests and collect results."""
    results = []
    for test in tests:
        print(f"Running test: {test['name']}...")
        output = run_agent_for_test(test_graph, test["input"])

        # Trajectory evaluation: exact match on tool name + parameters
        traj_ok = trajectory_match(output.tool_calls, test["expected_tool_calls"])
        resp_ok = response_match(output.final_response, test["expected_response_contains"])

        results.append({
            "name": test["name"],
            "trajectory_ok": traj_ok,
            "response_ok": resp_ok,
            "latency_ms": round(output.latency_ms, 2),
            "prompt_tokens": output.prompt_tokens,
            "response_tokens": output.response_tokens,
            "total_tokens": output.total_tokens,
            "tool_calls": [(tc.tool_name, tc.tool_parameters) for tc in output.tool_calls],
            "trace_id": output.trace_id,
            "final_response": output.final_response[:200] + "..." if len(output.final_response) > 200 else output.final_response
        })
        print(f"  Trajectory: {'✓' if traj_ok else '✗'}  Response: {'✓' if resp_ok else '✗'}")
        show_trace_link(output.trace_id)

    return results

# Run the tests
test_results = run_single_turn_tests(agent, AGENT_TESTS)
test_results

## Step 5: Run multi-turn tests

For a multi-turn test to mean anything, each turn must see the earlier turns of the same conversation. A compiled graph doesn't remember anything between calls on its own: the `agent` we used so far starts from an empty history every time.

We build a second agent with a **checkpointer**, which saves each conversation's messages under a `thread_id`, and send every turn of a test with the same `thread_id`. Each test gets a fresh thread id, so tests (and re-runs of this cell) don't share history.

In [ ]:
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver

# A checkpointer stores each thread's messages, so every turn sees the earlier ones.
multi_turn_agent: CompiledStateGraph = create_agent(
    model=llm_granite,
    tools=tools,
    checkpointer=InMemorySaver(),
)

In [ ]:
def run_multi_turn_tests(test_graph: CompiledStateGraph, tests: list[dict]) -> list[dict]:
    """Run multi-turn tests where each test is one conversation of several turns."""
    all_results = []

    for test in tests:
        print(f"\nRunning multi-turn test: {test['name']}")
        # One fresh thread per test: its turns share history, other tests (and re-runs) don't
        thread_id = f"{test['name']}-{uuid4()}"
        turn_results = []

        for i, turn in enumerate(test["turns"]):
            print(f"  Turn {i+1}: {turn['input'][:50]}...")
            output = run_agent_for_test(test_graph, turn["input"], thread_id=thread_id)

            # Trajectory evaluation: exact match on tool name + parameters
            traj_ok = trajectory_match(output.tool_calls, turn["expected_tool_calls"])
            resp_ok = response_match(output.final_response, turn["expected_response_contains"])

            turn_results.append({
                "input": turn["input"],
                "trajectory_ok": traj_ok,
                "response_ok": resp_ok,
                "tool_calls": [(tc.tool_name, tc.tool_parameters) for tc in output.tool_calls],
                "trace_id": output.trace_id,
                "final_response": output.final_response[:100] + "..." if len(output.final_response) > 100 else output.final_response,
            })
            print(f"    Trajectory: {'✓' if traj_ok else '✗'}  Response: {'✓' if resp_ok else '✗'}")
            show_trace_link(output.trace_id)

        all_results.append({"name": test["name"], "turns": turn_results})

    return all_results

# Run multi-turn tests
multi_turn_results = run_multi_turn_tests(multi_turn_agent, MULTI_TURN_TESTS)
multi_turn_results

## Step 6: Compute summary metrics

Summary metrics provide a high-level view of agent performance. Key metrics to track include:

- **Pass Rate**: Percentage of tests meeting all success criteria—the primary indicator of correctness
- **Average Latency**: Directly impacts user experience; watch for regressions after model updates
- **Average Tokens**: Input plus output tokens over every model call of a test, as reported by the model. Correlates with operational costs; evaluate if quality improvements justify increased cost

These metrics should be tracked over time to enable early detection of regressions.

In [ ]:
# Single-turn summary
passed = sum(1 for r in test_results if r["trajectory_ok"] and r["response_ok"])
total = len(test_results)
avg_latency = round(sum(r["latency_ms"] for r in test_results) / total, 2) if total > 0 else 0
avg_total_tokens = round(sum(r["total_tokens"] for r in test_results) / total, 2) if total > 0 else 0

single_turn_summary = {
    "passed": passed,
    "total": total,
    "pass_rate": f"{(passed/total)*100:.1f}%" if total > 0 else "N/A",
    "avg_latency_ms": avg_latency,
    "avg_total_tokens": avg_total_tokens
}

print("=" * 50)
print("SINGLE-TURN TEST SUMMARY")
print("=" * 50)
print(f"Tests Passed: {passed}/{total}")
print(f"Pass Rate: {single_turn_summary['pass_rate']}")
print(f"Average Latency: {avg_latency} ms")
print(f"Average Tokens: {avg_total_tokens}")
print()

# Multi-turn summary
multi_turn_passed = 0
multi_turn_total = 0

for test in multi_turn_results:
    for turn in test["turns"]:
        multi_turn_total += 1
        if turn["trajectory_ok"] and turn["response_ok"]:
            multi_turn_passed += 1

multi_turn_summary = {
    "passed": multi_turn_passed,
    "total": multi_turn_total,
    "pass_rate": f"{(multi_turn_passed/multi_turn_total)*100:.1f}%" if multi_turn_total > 0 else "N/A"
}

print("=" * 50)
print("MULTI-TURN TEST SUMMARY")
print("=" * 50)
print(f"Turns Passed: {multi_turn_passed}/{multi_turn_total}")
print(f"Pass Rate: {multi_turn_summary['pass_rate']}")
print()

# Overall summary
overall_passed = passed + multi_turn_passed
overall_total = total + multi_turn_total
print("=" * 50)
print("OVERALL TEST SUMMARY")
print("=" * 50)
print(f"Total Passed: {overall_passed}/{overall_total}")
print(f"Overall Pass Rate: {(overall_passed/overall_total)*100:.1f}%" if overall_total > 0 else "N/A")

## Step 7: Run the datasets as Langfuse experiments

Steps 4 to 6 print the results in the notebook: fine for one run, hard to compare across runs. A Langfuse **experiment**, introduced in [`02_Agent_Tracing_Observability.ipynb`](02_Agent_Tracing_Observability.ipynb), runs a dataset through the agent and stores every run with its scores. Because we run on the Langfuse datasets from Step 3, each run is attached to its dataset, so you can compare models, prompts or tool changes side by side in the Langfuse UI.

The experiment reuses the pieces built in this notebook:

| Experiment component | TDD piece |
| :--- | :--- |
| **Dataset** | `tdd-single-turn` and `tdd-multi-turn`, stored in Step 3 |
| **Task** | `run_agent_for_test`, nested in the experiment item's trace; for a conversation, one call per turn on the same thread |
| **Item evaluators** | `trajectory` and `response`: the share of turns that pass (1 or 0 for a single-turn test), plus `latency_ms` and `total_tokens` for the whole test |
| **Run evaluator** | `pass_rate`: the share of tests where every turn passes both the trajectory and the response check |

We set `max_concurrency=1` so the tests run one at a time, and the latency score isn't inflated by parallel runs competing for the same model.

In [ ]:
from langfuse import Evaluation


def to_output(result: AgentTestResult) -> dict:
    """What the experiment stores for one turn."""
    return {
        "final_response": result.final_response,
        "tool_calls": [{"tool_name": tc.tool_name, "tool_parameters": tc.tool_parameters} for tc in result.tool_calls],
        "latency_ms": result.latency_ms,
        "total_tokens": result.total_tokens,
    }


# Tasks: run one dataset item, nested in the experiment item's trace
def single_turn_task(*, item, **kwargs):
    return to_output(run_agent_for_test(agent, item.input, new_trace=False))


def multi_turn_task(*, item, **kwargs):
    thread_id = f"{item.metadata['name']}-{uuid4()}"
    return [
        to_output(run_agent_for_test(multi_turn_agent, turn_input, thread_id=thread_id, new_trace=False))
        for turn_input in item.input["turns"]
    ]


def turns(output, expected_output) -> list[tuple[dict, dict]]:
    """Pair each turn's output with its expectations; a single-turn test is one turn."""
    if isinstance(output, list):
        return list(zip(output, expected_output))
    return [(output, expected_output)]


# Item-level evaluators: the same checks as Steps 4 to 6, recorded as Langfuse scores
def trajectory_evaluator(*, output, expected_output, **kwargs):
    pairs = turns(output, expected_output)
    ok = sum(trajectory_match([ToolCall(**tc) for tc in out["tool_calls"]], exp["tool_calls"]) for out, exp in pairs)
    return Evaluation(name="trajectory", value=ok / len(pairs), comment=f"{ok}/{len(pairs)} turns with the expected tool calls")


def response_evaluator(*, output, expected_output, **kwargs):
    pairs = turns(output, expected_output)
    ok = sum(response_match(out["final_response"], exp["response_contains"]) for out, exp in pairs)
    return Evaluation(name="response", value=ok / len(pairs), comment=f"{ok}/{len(pairs)} turns with the expected content")


def latency_evaluator(*, output, expected_output, **kwargs):
    return Evaluation(name="latency_ms", value=round(sum(out["latency_ms"] for out, _ in turns(output, expected_output)), 2))


def tokens_evaluator(*, output, expected_output, **kwargs):
    return Evaluation(name="total_tokens", value=sum(out["total_tokens"] for out, _ in turns(output, expected_output)))


# Run-level evaluator: a test passes only if every turn passes both checks
def pass_rate_evaluator(*, item_results, **kwargs):
    def passed(result):
        scores = {e.name: e.value for e in result.evaluations}
        return scores.get("trajectory") == 1.0 and scores.get("response") == 1.0

    passed_count = sum(passed(r) for r in item_results)
    total = len(item_results)
    return Evaluation(
        name="pass_rate",
        value=passed_count / total if total else 0.0,
        comment=f"{passed_count}/{total} tests passed",
    )


if not LANGFUSE_AVAILABLE:
    print("Langfuse unavailable: skipping the experiments")
else:
    for dataset_name, task in [(SINGLE_TURN_DATASET, single_turn_task), (MULTI_TURN_DATASET, multi_turn_task)]:
        experiment = langfuse_client.get_dataset(dataset_name).run_experiment(
            name="TDD",
            description=f"{dataset_name} on {llm_granite.model}",
            task=task,
            evaluators=[trajectory_evaluator, response_evaluator, latency_evaluator, tokens_evaluator],
            run_evaluators=[pass_rate_evaluator],
            max_concurrency=1,
            metadata={"model": llm_granite.model},
        )
        langfuse_client.flush()

        print(experiment.format())
        show_link(f"View the {dataset_name} experiment run in Langfuse", experiment.dataset_run_url)
        for item_result in experiment.item_results:
            show_trace_link(item_result.trace_id, label=f"View {item_result.item.metadata['name']} in Langfuse")

In Langfuse, open **Datasets**, then `tdd-single-turn` or `tdd-multi-turn`: each experiment run is listed with its scores. Each test is a trace holding the `TDD` agent runs (the model generations and tool calls) and the four item scores; the run also gets the `pass_rate` score. Run the experiments again after changing the model, the prompt or a tool, and compare the runs.

## Summary

In this recipe, you learned how to build a structured testing framework for function-calling agents:

1. **Defined evaluation helpers** for trajectory (tool calls) and response validation
2. **Created a test runner** that captures tool calls, responses, and performance metrics
3. **Wrote structured test cases** covering single-turn and multi-turn interactions
4. **Computed summary metrics** including pass rates, latency, and token usage
5. **Traced every test run in Langfuse**, **stored the test sets as Langfuse datasets** and **ran them as Langfuse experiments**, with the checks recorded as scores

### Next steps

- **Expand your test set** with edge cases and failure scenarios (e.g., invalid inputs, non-existent cities)
- **Add more tools** to test complex multi-tool interactions
- **Implement LLM-as-a-judge** for response evaluation when substring matching isn't sufficient
- **Integrate with CI/CD** to run tests automatically on every code change
- **Compare model performance** by running the same tests with different models
- **Track metrics over time** by re-running the Langfuse experiments and comparing their runs on each dataset

For more on agent evaluation concepts and approaches, see the workshop guide: [4.1 Functional Testing](https://ibm-granite-community.github.io/granite-agent-workshop/part-04-testing-agents/functional-testing/).